# 📚 Data Cleaning
### 데이터 정제

> **Section 5 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA  
> 전체 11개 섹션 중 **5번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to handle missing values with `dropna` / `fillna` / `ffill` / `bfill` / `interpolate`, and remove duplicates with `drop_duplicates`  
`dropna` / `fillna` / `ffill` / `bfill` / `interpolate`로 결측치를 처리하고, `drop_duplicates`로 중복을 제거하는 방법
- [x] How to convert types safely with `astype` / `to_numeric` / `to_datetime`, and clean text with the `.str` accessor  
`astype` / `to_numeric` / `to_datetime`으로 안전하게 타입을 변환하고, `.str` accessor로 텍스트를 정제하는 방법
- [x] How to check conditions across rows/columns with `any()` / `all()`, and cap or replace values with `clip()` / `where()` / `mask()`  
`any()` / `all()`로 행·열에 걸쳐 조건을 확인하고, `clip()` / `where()` / `mask()`로 값을 제한하거나 교체하는 방법

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
Data cleaning is the step where you actually *fix* everything EDA found wrong: missing values get filled or dropped, duplicate rows get removed, columns with the wrong type get converted, messy text gets trimmed and normalized, and outliers get capped. It's usually the single longest step in any real analysis.

**한글**
데이터 정제는 EDA가 발견한 문제를 실제로 *고치는* 단계입니다: 결측치를 채우거나 삭제하고, 중복 행을 제거하고, 타입이 잘못된 열을 변환하고, 지저분한 텍스트를 다듬어 정규화하고, 이상치를 제한합니다. 보통 실제 분석에서 가장 오래 걸리는 단일 단계입니다.

## Why do we use it?
*(When is it useful?)*

**English**
A number computed on dirty data is worse than no number at all — it *looks* correct, so nobody questions it, until someone discovers the average included three duplicate rows, or a stray `"-"` silently turned an entire column into text. Cleaning turns "the data I was handed" into "the data I can actually trust a calculation on."

**한글**
지저분한 데이터로 계산한 숫자는 숫자가 아예 없는 것보다 더 나쁩니다 — *맞아 보이기* 때문에 아무도 의심하지 않다가, 나중에야 평균에 중복 행 3개가 포함되어 있었다거나 `"-"` 하나가 열 전체를 조용히 텍스트로 바꿔놨다는 걸 발견합니다. 정제는 "받은 데이터"를 "계산을 실제로 믿을 수 있는 데이터"로 바꿉니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
Industry folklore says 60-80% of analysis time goes here — and it's true. Every recurring report (weekly sales, monthly headcount) needs the same cleaning pipeline re-applied to a fresh, differently-messy export every single time, which is exactly why building a *repeatable* pipeline (Example 8 below) matters more than cleaning any single dataset by hand.

**한글**
업계에서 흔히 하는 말로 분석 시간의 60~80%가 여기에 쓰인다고 하는데 — 사실입니다. 매주 반복되는 매출 보고서, 매달 반복되는 인원 현황처럼 반복되는 모든 보고서는 매번 새롭게, 조금씩 다르게 지저분한 내보내기 파일에 똑같은 정제 파이프라인을 다시 적용해야 합니다 — 그래서 단일 데이터셋 하나를 손으로 정제하는 것보다 *반복 가능한* 파이프라인(아래 Example 8)을 만드는 것이 더 중요합니다.

### Quick Comparison: JS/TS vs pandas / 빠른 비교

| Concept / 개념 | JavaScript / TypeScript | pandas |
|---|---|---|
| Remove missing entries / 결측 항목 제거 | `arr.filter(x => x != null)` | `df.dropna()` |
| Fill missing entries / 결측 항목 채우기 | `arr.map(x => x ?? default)` | `df["col"].fillna(default)` |
| Remove duplicate objects / 중복 객체 제거 | manual `Set` + `JSON.stringify` / 직접 `Set` + `JSON.stringify` | `df.drop_duplicates()` |
| String → number / 문자열 → 숫자 | `Number(str)` / `parseInt(str)` | `df["col"].astype(int)` / `pd.to_numeric()` |
| String → date / 문자열 → 날짜 | `new Date(str)` | `pd.to_datetime()` |
| Trim + lowercase every item / 전체 trim + 소문자화 | `arr.map(s => s.trim().toLowerCase())` | `df["col"].str.strip().str.lower()` |
| Cap a value into a range / 값을 범위 안으로 제한 | `Math.min(Math.max(x, lo), hi)` | `df["col"].clip(lo, hi)` |

---
# 📝 Syntax

## Basic Syntax

In [1]:
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    "order_id": [101, 102, 103],
    "customer": ["Alice", None, "Charlie"],
    "amount": [45000, 32000, None],
})

# Two of the most common cleaning moves -- drop or fill missing values
# 가장 흔한 정제 작업 두 가지 -- 결측치를 삭제하거나 채우기
print("dropna():")
print(orders.dropna())
print()

print("fillna(...):")
print(orders.fillna({"customer": "Unknown", "amount": 0}))

dropna():
   order_id customer   amount
0       101    Alice  45000.0

fillna(...):
   order_id customer   amount
0       101    Alice  45000.0
1       102  Unknown  32000.0
2       103  Charlie      0.0


## Common Variations

In [2]:
import pandas as pd

df = pd.DataFrame({"amount_str": ["45,000", "32,000", "abc"]})

# Convert text to numbers, turning anything unconvertible into NaN instead of crashing
# 텍스트를 숫자로 변환, 변환 불가능한 값은 오류 대신 NaN으로 처리
df["amount"] = pd.to_numeric(df["amount_str"].str.replace(",", ""), errors="coerce")
print(df)
print()

# Clean up messy text in one chain / 지저분한 텍스트를 한 번에 정리
names = pd.Series([" Alice ", "BOB", " charlie"])
print(names.str.strip().str.title())

  amount_str   amount
0     45,000  45000.0
1     32,000  32000.0
2        abc      NaN

0      Alice
1        Bob
2    Charlie
dtype: str


---
# 🧪 Small Examples

## Example 1 — Handling Missing Values: dropna / fillna / ffill / bfill / interpolate
*(Covers source section 5-1)*

**English:** `dropna()` deletes rows with missing values — `subset=[...]` limits the check to columns that actually matter, and `thresh=N` only drops a row if it has fewer than N valid (non-missing) values. `fillna()` replaces missing values instead of deleting rows, and usually needs a *different* strategy per column. For ordered data (like a daily time series), `ffill()` / `bfill()` / `interpolate()` fill gaps using neighboring values instead of a fixed constant.  
**한글:** `dropna()`는 결측치가 있는 행을 삭제합니다 — `subset=[...]`은 실제로 중요한 열만 확인하도록 제한하고, `thresh=N`은 유효한(결측이 아닌) 값이 N개 미만인 행만 삭제합니다. `fillna()`는 행을 삭제하는 대신 결측치를 채우는데, 보통 열마다 *다른* 전략이 필요합니다. 순서가 있는 데이터(일별 시계열 등)라면 `ffill()` / `bfill()` / `interpolate()`가 고정값 대신 이웃 값으로 빈틈을 채웁니다.

In [15]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer": ["Alice", None, "Charlie", "Diana", None],
    "amount": [45000, 32000, None, 28000, 61000],
    "region": ["Seoul", "Busan", "Seoul", None, "Incheon"],
})
print("original:")
print(orders)
print()

print("dropna() -- any row with ANY missing value is gone:")
print(orders.dropna())
print()

print("dropna(subset=['customer']) -- only checks 'customer':")
print(orders.dropna(subset=["customer"]))
print()

print("dropna(thresh=3) -- keep rows with at least 3 valid values:")
print(orders.dropna(thresh=3))
print()

# fillna -- a different strategy per column / fillna -- 열마다 다른 전략
cleaned = orders.copy()
cleaned["customer"] = cleaned["customer"].fillna("Unknown")
cleaned["amount"] = cleaned["amount"].fillna(cleaned["amount"].mean())
print("fillna() with a per-column strategy:")
print(cleaned)
print()

# ffill / bfill / interpolate -- for ordered data / 순서가 있는 데이터를 위한 ffill / bfill / interpolate
sales_daily = pd.DataFrame({"day": [1, 2, 3, 4, 5], "sales": [100, None, None, 400, 500]})
sales_daily["ffill"] = sales_daily["sales"].ffill()
sales_daily["bfill"] = sales_daily["sales"].bfill()
sales_daily["interp"] = sales_daily["sales"].interpolate()
print("ffill / bfill / interpolate on ordered data:")
print(sales_daily)

original:
   order_id customer   amount   region
0       101    Alice  45000.0    Seoul
1       102      NaN  32000.0    Busan
2       103  Charlie      NaN    Seoul
3       104    Diana  28000.0      NaN
4       105      NaN  61000.0  Incheon

dropna() -- any row with ANY missing value is gone:
   order_id customer   amount region
0       101    Alice  45000.0  Seoul

dropna(subset=['customer']) -- only checks 'customer':
   order_id customer   amount region
0       101    Alice  45000.0  Seoul
2       103  Charlie      NaN  Seoul
3       104    Diana  28000.0    NaN

dropna(thresh=3) -- keep rows with at least 3 valid values:
   order_id customer   amount   region
0       101    Alice  45000.0    Seoul
1       102      NaN  32000.0    Busan
2       103  Charlie      NaN    Seoul
3       104    Diana  28000.0      NaN
4       105      NaN  61000.0  Incheon

fillna() with a per-column strategy:
   order_id customer   amount   region
0       101    Alice  45000.0    Seoul
1       102  U

## Example 2 — Removing Duplicates: drop_duplicates
*(Covers source section 5-2)*

**English:** `drop_duplicates()` with no arguments only removes rows that are identical across *every* column. `subset=[...]` checks just the columns that define "the same real-world thing" (ignoring an auto-incrementing ID, for example), and `keep="first"` / `"last"` / `False` control which copy survives — or whether *all* copies get removed.  
**한글:** 인수 없는 `drop_duplicates()`는 *모든* 열에 걸쳐 완전히 동일한 행만 제거합니다. `subset=[...]`은 "같은 실제 대상"을 정의하는 열만 확인하고(예를 들어 자동 증가하는 ID는 무시), `keep="first"` / `"last"` / `False`는 어느 사본이 살아남는지 — 또는 *모든* 사본을 제거할지 — 를 결정합니다.

In [4]:
import pandas as pd

reviews = pd.DataFrame({
    "user_id": [1, 2, 1, 3, 2, 4],
    "product": ["A", "B", "A", "C", "B", "A"],
    "rating": [5, 4, 5, 3, 4, 4],
    "comment": ["Good", "OK", "Good", "Bad", "OK", "Great"],
})
print("original:")
print(reviews)
print()

print("drop_duplicates() -- exact full-row matches only:")
print(reviews.drop_duplicates())
print()

print("drop_duplicates(subset=['user_id','product']) -- first copy kept:")
print(reviews.drop_duplicates(subset=["user_id", "product"]))
print()

print("keep='last' -- keep the most recent entry instead:")
print(reviews.drop_duplicates(subset=["user_id", "product"], keep="last"))
print()

print("keep=False -- drop every row involved in a duplicate:")
print(reviews.drop_duplicates(subset=["user_id", "product"], keep=False))

original:
   user_id product  rating comment
0        1       A       5    Good
1        2       B       4      OK
2        1       A       5    Good
3        3       C       3     Bad
4        2       B       4      OK
5        4       A       4   Great

drop_duplicates() -- exact full-row matches only:
   user_id product  rating comment
0        1       A       5    Good
1        2       B       4      OK
3        3       C       3     Bad
5        4       A       4   Great

drop_duplicates(subset=['user_id','product']) -- first copy kept:
   user_id product  rating comment
0        1       A       5    Good
1        2       B       4      OK
3        3       C       3     Bad
5        4       A       4   Great

keep='last' -- keep the most recent entry instead:
   user_id product  rating comment
2        1       A       5    Good
3        3       C       3     Bad
4        2       B       4      OK
5        4       A       4   Great

keep=False -- drop every row involved in a duplic

## Example 3 — Type Conversion: astype / to_numeric / to_datetime
*(Covers source section 5-3)*

**English:** `astype()` converts strictly — the moment it hits one value it can't convert, it raises an error and stops the whole cell. `pd.to_numeric()` and `pd.to_datetime()` with `errors="coerce"` are the lenient alternative: anything unconvertible quietly becomes `NaN` / `NaT` instead of crashing. `format=` handles dates that aren't in the standard `YYYY-MM-DD` shape.  
**한글:** `astype()`은 엄격하게 변환합니다 — 변환할 수 없는 값을 하나라도 만나면 오류를 내며 셀 전체를 멈춥니다. `errors="coerce"`를 붙인 `pd.to_numeric()`과 `pd.to_datetime()`은 더 관대한 대안입니다 — 변환 불가능한 값은 오류 대신 조용히 `NaN` / `NaT`가 됩니다. `format=`은 표준 `YYYY-MM-DD` 형태가 아닌 날짜를 처리합니다.

In [5]:
import pandas as pd

# astype -- strict, fails loudly the instant it can't convert something
# astype -- 엄격함, 변환할 수 없는 값을 만나는 순간 크게 실패함
df = pd.DataFrame({"user_id": ["001", "002", "003"], "active": [1, 0, 1]})
df["user_id"] = df["user_id"].astype(int)
df["active"] = df["active"].astype(bool)
print(df)
print()

try:
    pd.Series(["45000", "abc"]).astype(int)
except ValueError as e:
    print("astype(int) on 'abc' raises:", e)
print()

# to_numeric(errors="coerce") -- lenient, bad values become NaN instead of crashing
# to_numeric(errors="coerce") -- 관대함, 잘못된 값은 오류 대신 NaN이 됨
amounts = pd.Series(["45000", "32000", "abc", "61000"])
print("to_numeric with errors='coerce':")
print(pd.to_numeric(amounts, errors="coerce"))
print()

# to_datetime(errors="coerce") -- same idea, for dates / to_datetime(errors="coerce") -- 날짜에 대해 같은 개념
signup = pd.DataFrame({"date_str": ["2024-01-15", "2024-03-22", "bad_date"]})
signup["date"] = pd.to_datetime(signup["date_str"], errors="coerce")
print(signup)
print()

# format= -- for dates that aren't in the standard shape / format= -- 표준 형태가 아닌 날짜
nonstd = pd.DataFrame({"date_str": ["15/01/2024", "22/03/2024", "08/07/2024"]})
nonstd["date"] = pd.to_datetime(nonstd["date_str"], format="%d/%m/%Y")
print(nonstd)

   user_id  active
0        1    True
1        2   False
2        3    True

astype(int) on 'abc' raises: invalid literal for int() with base 10: 'abc'

to_numeric with errors='coerce':
0    45000.0
1    32000.0
2        NaN
3    61000.0
dtype: float64

     date_str       date
0  2024-01-15 2024-01-15
1  2024-03-22 2024-03-22
2    bad_date        NaT

     date_str       date
0  15/01/2024 2024-01-15
1  22/03/2024 2024-03-22
2  08/07/2024 2024-07-08


## Example 4 — Dropping & Renaming: drop() / rename() / Bulk Column Cleanup
*(Covers source section 5-4)*

**English:** `drop(columns=[...])` removes columns, `drop(index=[...])` removes rows by their index label. `rename(columns={...})` renames specific columns one at a time. When *every* column name is messy (mixed case, spaces, parentheses), a chained `.str` cleanup on `df.columns` fixes them all in four lines.  
**한글:** `drop(columns=[...])`는 열을 제거하고, `drop(index=[...])`는 인덱스 라벨로 행을 제거합니다. `rename(columns={...})`는 특정 열의 이름을 하나씩 바꿉니다. *모든* 열 이름이 지저분할 때(대소문자 혼합, 공백, 괄호), `df.columns`에 체이닝된 `.str` 정리를 적용하면 네 줄로 전부 고칠 수 있습니다.

In [6]:
import pandas as pd

df = pd.DataFrame({
    "Order ID": [101, 102, 103],
    "Customer Name": ["Alice", "Bob", "Charlie"],
    "Amount (KRW)": [45000, 32000, 61000],
    "tmp_col": ["x", "y", "z"],
    "internal": [1, 2, 3],
})

# Drop columns / 열 삭제
print("drop(columns=...):")
print(df.drop(columns=["tmp_col", "internal"]))
print()

# Drop rows by index / 인덱스로 행 삭제
print("drop(index=...):")
print(df.drop(index=[0, 2]))
print()

# Rename specific columns / 특정 열 이름 변경
print("rename(columns=...):")
print(df.rename(columns={"Order ID": "order_id", "Customer Name": "customer"}))
print()

# Bulk cleanup -- fix EVERY column name at once / 일괄 정리 -- 모든 열 이름을 한 번에 수정
small = pd.DataFrame({"Order ID": [101], "Customer Name": ["Alice"], "Amount (KRW)": [45000]})
small.columns = (
    small.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[()]+", "", regex=True)
    .str.strip("_")
)
print("after bulk column-name cleanup:", small.columns.tolist())
print(small)

drop(columns=...):
   Order ID Customer Name  Amount (KRW)
0       101         Alice         45000
1       102           Bob         32000
2       103       Charlie         61000

drop(index=...):
   Order ID Customer Name  Amount (KRW) tmp_col  internal
1       102           Bob         32000       y         2

rename(columns=...):
   order_id customer  Amount (KRW) tmp_col  internal
0       101    Alice         45000       x         1
1       102      Bob         32000       y         2
2       103  Charlie         61000       z         3

after bulk column-name cleanup: ['order_id', 'customer_name', 'amount_krw']
   order_id customer_name  amount_krw
0       101         Alice       45000


## Example 5 — str Accessor: Cleaning Text Columns
*(Covers source section 5-5)*

**English:** The `.str` accessor applies a string method to every value in a column at once — `.strip()` trims whitespace, `.lower()` / `.title()` normalize casing, `.replace()` removes unwanted substrings, `.contains()` filters by a substring, and `.split(expand=True)` / `.extract()` pull structured pieces out of a single messy text column.  
**한글:** `.str` accessor는 열의 모든 값에 문자열 메서드를 한 번에 적용합니다 — `.strip()`은 공백을 제거하고, `.lower()` / `.title()`은 대소문자를 정규화하고, `.replace()`는 원치 않는 부분 문자열을 제거하고, `.contains()`는 부분 문자열로 필터링하며, `.split(expand=True)` / `.extract()`는 지저분한 텍스트 열 하나에서 구조화된 조각을 뽑아냅니다.

### `.str` Accessor Quick Reference / 빠른 참조

| Method / 메서드 | Role / 역할 | Example / 예시 |
|---|---|---|
| `.str.strip()` | trim whitespace / 양쪽 공백 제거 | `" a "` → `"a"` |
| `.str.lower()` / `.str.title()` | normalize casing / 대소문자 정규화 | `"BOB"` → `"bob"` / `"Bob"` |
| `.str.replace(a, b)` | substitute text / 텍스트 치환 | `"SalesTeam"` → `"Sales"` |
| `.str.contains("x")` | substring filter (bool) / 부분 문자열 포함 여부 | `True` / `False` |
| `.str.split(sep, expand=True)` | split into separate columns / 분리해서 여러 열로 | one column → many / 한 열 → 여러 열 |
| `.str.extract(r"pattern")` | pull out a regex match / 정규식으로 패턴 추출 | matched text / 매칭된 텍스트 |
| `.str.len()` | string length / 문자열 길이 | `5` |
| `.str.zfill(n)` | zero-pad on the left / 앞에 0 채우기 | `"42"` → `"00042"` |

In [7]:
import pandas as pd

contacts = pd.DataFrame({
    "name": [" Minsu ", "Younghee", " Junho"],
    "email": ["KIM@COMPANY.COM", "lee.younghee@corp.kr", "park@COMPANY.COM"],
    "phone": ["010-1234-5678", "010-9876-5432", "010-5555-1234"],
    "dept": ["SalesTeam", "MarketingTeam", "EngineeringTeam"],
})

# strip / lower / replace / 공백 제거 / 소문자화 / 치환
contacts["name"] = contacts["name"].str.strip()
contacts["email"] = contacts["email"].str.lower()
contacts["dept"] = contacts["dept"].str.replace("Team", "", regex=False)
print(contacts)
print()

# contains -- filter rows by a substring / contains -- 부분 문자열로 행 필터링
print("emails containing 'company':")
print(contacts[contacts["email"].str.contains("company")])
print()

# split(expand=True) -- one column becomes several / split(expand=True) -- 한 열이 여러 열로
phone_parts = contacts["phone"].str.split("-", expand=True)
phone_parts.columns = ["area", "mid", "last"]
print(phone_parts)
print()

# extract -- pull a regex match out of each value / extract -- 정규식으로 각 값에서 패턴 추출
print("everything before the @ in each email:")
print(contacts["email"].str.extract(r"^(.+)@"))

       name                 email          phone         dept
0     Minsu       kim@company.com  010-1234-5678        Sales
1  Younghee  lee.younghee@corp.kr  010-9876-5432    Marketing
2     Junho      park@company.com  010-5555-1234  Engineering

emails containing 'company':
    name             email          phone         dept
0  Minsu   kim@company.com  010-1234-5678        Sales
2  Junho  park@company.com  010-5555-1234  Engineering

  area   mid  last
0  010  1234  5678
1  010  9876  5432
2  010  5555  1234

everything before the @ in each email:
              0
0           kim
1  lee.younghee
2          park


## Example 6 — any() / all(): Row-wise vs Column-wise Checks
*(Covers source section 5-6)*

**English:** `.any()` / `.all()` answer "is at least one True?" / "are they all True?" — but the direction matters. `axis=0` (the default) checks **down each column**, giving one answer per column. `axis=1` checks **across each row**, giving one answer per row — the shape you need to build a filter mask like "any subject below 60."  
**한글:** `.any()` / `.all()`은 "적어도 하나는 True인가?" / "전부 True인가?"에 답하는데 — 방향이 중요합니다. `axis=0`(기본값)은 **각 열을 따라 아래로** 확인해서 열마다 하나의 답을 줍니다. `axis=1`은 **각 행을 가로질러** 확인해서 행마다 하나의 답을 주는데 — "어느 과목이든 60점 미만"같은 필터 마스크를 만들 때 필요한 형태입니다.

### `axis` Direction / 방향 정리

| | `axis=0` (default) | `axis=1` |
|---|---|---|
| Direction / 방향 | down each column / 열 방향 (↓) | across each row / 행 방향 (→) |
| Result / 결과 | one value per column / 열마다 값 하나 | one value per row / 행마다 값 하나 |
| Typical use / 주 용도 | "which columns have a problem?" / "어느 열에 문제가?" | "which rows meet the condition?" / "어느 행이 조건 충족?" |

In [8]:
import pandas as pd

scores = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "math": [85, 40, 92, 78],
    "eng": [70, 55, 88, 45],
    "sci": [90, 62, 75, 80],
})
num_cols = scores[["math", "eng", "sci"]]

# axis=0 -- down each column: does THIS SUBJECT have anyone below 60?
# axis=0 -- 열 방향: 이 과목에서 60점 미만인 사람이 있는가?
print("(num_cols < 60).any(axis=0):")
print((num_cols < 60).any(axis=0))
print()

# axis=1 -- across each row: does THIS STUDENT have any subject below 60?
# axis=1 -- 행 방향: 이 학생이 60점 미만인 과목이 하나라도 있는가?
fail_mask = (num_cols < 60).any(axis=1)
print("students with at least one subject below 60:")
print(scores[fail_mask])
print()

# .all(axis=1) -- every subject has to clear the bar / .all(axis=1) -- 모든 과목이 기준을 넘어야 함
pass_mask = (num_cols >= 75).all(axis=1)
print("students with EVERY subject at 75+:")
print(scores[pass_mask])
print()

# The classic combo: "does this table have ANY missing value at all?"
# 자주 쓰는 조합: "이 테이블에 결측치가 하나라도 있는가?"
messy = pd.DataFrame({"name": ["A", "B", "C"], "score": [85, None, 92], "grade": ["A", None, "A"]})
print("isna().any() -- per column:")
print(messy.isna().any())
print("isna().any().any() -- one final answer for the whole table:", messy.isna().any().any())

(num_cols < 60).any(axis=0):
math     True
eng      True
sci     False
dtype: bool

students with at least one subject below 60:
    name  math  eng  sci
1    Bob    40   55   62
3  Diana    78   45   80

students with EVERY subject at 75+:
      name  math  eng  sci
2  Charlie    92   88   75

isna().any() -- per column:
name     False
score     True
grade     True
dtype: bool
isna().any().any() -- one final answer for the whole table: True


## Example 7 — clip() / where() / mask(): Capping & Conditional Replace
*(Covers source section 5-7)*

**English:** All three touch values based on a condition, but differently. `.clip(lo, hi)` pushes out-of-range values back to the nearest boundary — the standard way to cap outliers without deleting the row. `.where(cond, other)` keeps values where `cond` is `True` and replaces the rest. `.mask(cond, other)` is the mirror image — it replaces where `cond` is `True` and keeps the rest.   
**한글:** 셋 다 조건에 따라 값을 다루지만 방식이 다릅니다. `.clip(lo, hi)`는 범위를 벗어난 값을 가장 가까운 경계값으로 밀어넣습니다 — 행을 삭제하지 않고 이상치를 제한하는 표준 방법입니다. `.where(cond, other)`는 `cond`가 `True`인 곳은 그대로 두고 나머지를 교체합니다. `.mask(cond, other)`는 그 반대입니다 — `cond`가 `True`인 곳을 교체하고 나머지는 그대로 둡니다.

### Three Methods Compared / 세 메서드 비교

| Method / 메서드 | Where `cond` is True / 조건 True인 곳 | Where `cond` is False / 조건 False인 곳 | Typical use / 주 용도 |
|---|---|---|---|
| `clip(lo, hi)` | kept, if in range / 범위 안이면 유지 | pushed to the nearest boundary / 가장 가까운 경계로 이동 | outlier capping / 이상치 캡핑 |
| `where(cond, other)` | kept as-is / 그대로 유지 | replaced with `other` / `other`로 교체 | "keep only what matches" / "조건 충족만 보존" |
| `mask(cond, other)` | replaced with `other` / `other`로 교체 | kept as-is / 그대로 유지 | "remove only what matches" / "조건 충족만 제거" |

In [16]:
import pandas as pd

temps = pd.DataFrame({
    "city": ["Seoul", "Busan", "Daegu", "Incheon", "Gwangju"],
    "temp": [-15, 5, 38, 2, 42],
})

# clip -- push out-of-range values to the nearest boundary (0 to 35) / clip -- 범위 밖 값을 경계값으로
print("clip(0, 35):")
print(temps["temp"].clip(0, 35))
print()

# where -- keep values >= 0 as-is, replace the rest with 0 / where -- 0 이상은 유지, 나머지는 0으로 교체
print("where(temp >= 0, other=0):")
print(temps["temp"].where(temps["temp"] >= 0, other=0))
print()

# mask -- replace values > 35 with 35, keep the rest / mask -- 35 초과는 35로 교체, 나머지는 유지
print("mask(temp > 35, other=35):")
print(temps["temp"].mask(temps["temp"] > 35, other=35))
print()

# np.where -- different purpose: BUILD a new column from a condition, not modify an existing one
# np.where -- 다른 목적: 기존 값을 수정하는 게 아니라 조건으로 새 열을 만듦
temps["status"] = np.where(temps["temp"] > 30, "Heatwave", "Normal")
print(temps)

clip(0, 35):
0     0
1     5
2    35
3     2
4    35
Name: temp, dtype: int64

where(temp >= 0, other=0):
0     0
1     5
2    38
3     2
4    42
Name: temp, dtype: int64

mask(temp > 35, other=35):
0   -15
1     5
2    35
3     2
4    35
Name: temp, dtype: int64

      city  temp    status
0    Seoul   -15    Normal
1    Busan     5    Normal
2    Daegu    38  Heatwave
3  Incheon     2    Normal
4  Gwangju    42  Heatwave


## Example 8 — Common Combo: Full Post-Load Cleaning Pipeline
*(Covers source section 5-8)*

**English:** This is the pipeline that runs right after almost every real `read_csv()`: clean the column names first (so every later line can refer to them cleanly), normalize text columns, convert types safely, and deduplicate last (since dtype/text issues can hide true duplicates from being detected).  
**한글:** 이는 거의 모든 실제 `read_csv()` 직후에 실행되는 파이프라인입니다: 열 이름을 먼저 정리하고(이후 모든 줄에서 깔끔하게 참조할 수 있도록), 텍스트 열을 정규화하고, 타입을 안전하게 변환한 뒤, 마지막에 중복을 제거합니다(dtype·텍스트 문제가 있으면 진짜 중복이 감지되지 않을 수 있으므로).

In [10]:
import pandas as pd
from io import StringIO

raw_csv = '''Order ID,Customer Name,Amount (KRW),Signup Date,Active
001, Alice ,"45,000",2024-01-15,1
002,BOB,"32,000",bad-date,0
003, Charlie ,"61,000",2024-07-08,1
001, Alice ,"45,000",2024-01-15,1'''

df = pd.read_csv(StringIO(raw_csv), thousands=",")
print("right after loading -- 4 problems visible:")
print(df)
print()

# Step 1 -- clean column names / 1단계 -- 열 이름 정리
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[()]+", "", regex=True)
    .str.strip("_")
)

# Step 2 -- normalize text columns / 2단계 -- 텍스트 열 정규화
df["customer_name"] = df["customer_name"].str.strip().str.title()

# Step 3 -- convert types safely / 3단계 -- 타입을 안전하게 변환
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
df["active"] = df["active"].astype(bool)

# Step 4 -- deduplicate last / 4단계 -- 마지막에 중복 제거
df = df.drop_duplicates(subset=["order_id", "customer_name"])

print("after the full pipeline:")
print(df)
print(df.dtypes)
print()

# Final check -- exactly one intentional NaT remains (the unparseable "bad-date")
# 최종 확인 -- 의도된 NaT 하나만 남음 (파싱 불가능했던 "bad-date")
print("any missing values left?", df.isna().any().any())
print(df.isna().sum())

right after loading -- 4 problems visible:
   Order ID Customer Name  Amount (KRW) Signup Date  Active
0         1        Alice          45000  2024-01-15       1
1         2           BOB         32000    bad-date       0
2         3      Charlie          61000  2024-07-08       1
3         1        Alice          45000  2024-01-15       1

after the full pipeline:
   order_id customer_name  amount_krw signup_date  active
0         1         Alice       45000  2024-01-15    True
1         2           Bob       32000         NaT   False
2         3       Charlie       61000  2024-07-08    True
order_id                  int64
customer_name               str
amount_krw                int64
signup_date      datetime64[us]
active                     bool
dtype: object

any missing values left? True
order_id         0
customer_name    0
amount_krw       0
signup_date      1
active           0
dtype: int64


## Example 9 (Practice) — Fill in the Blanks
*(Based on the practice exercise in source section 5-9)*

**English:** Fill in each `________` blank below to build a cleaning pipeline for this messy employee roster. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워서 이 지저분한 직원 명단을 위한 정제 파이프라인을 완성하세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [17]:
import pandas as pd
from io import StringIO

csv_text = '''name,dept,salary,hire_date
 alice ,SalesTeam,"48,000,000",2022-03-15
BOB,MarketingTeam,"35,000,000",bad_date
 alice ,SalesTeam,"48,000,000",2022-03-15
charlie,EngineeringTeam,"62,000,000",2021-11-01'''

df = pd.read_csv(StringIO(csv_text), thousands=",")

# 1. name -- strip whitespace + title case / 공백 제거 + 첫 글자만 대문자
df["name"] = df["name"].str.strip().str.title()

# 2. dept -- remove the "Team" suffix / "Team" 접미사 제거
df["dept"] = df["dept"].str.replace("Team", "", regex=False)

# 3. hire_date -- convert to datetime, bad values become NaT / datetime으로 변환, 잘못된 값은 NaT
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")

# 4. remove duplicate employees (same name + dept) / 중복 직원 제거 (name + dept 기준)
df = df.drop_duplicates(subset=["name", "dept"])

print(df)
print(df.dtypes)

      name         dept    salary  hire_date
0    Alice        Sales  48000000 2022-03-15
1      Bob    Marketing  35000000        NaT
3  Charlie  Engineering  62000000 2021-11-01
name                    str
dept                    str
salary                int64
hire_date    datetime64[us]
dtype: object


### 💡 Hint / 힌트
`strip` · `title` · `regex` · `coerce` · `duplicates`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd
from io import StringIO

csv_text = '''name,dept,salary,hire_date
 alice ,SalesTeam,"48,000,000",2022-03-15
BOB,MarketingTeam,"35,000,000",bad_date
 alice ,SalesTeam,"48,000,000",2022-03-15
charlie,EngineeringTeam,"62,000,000",2021-11-01'''

df = pd.read_csv(StringIO(csv_text), thousands=",")

df["name"] = df["name"].str.strip().str.title()
df["dept"] = df["dept"].str.replace("Team", "", regex=False)
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")
df = df.drop_duplicates(subset=["name", "dept"])

print(df)
print(df.dtypes)

# Down to 3 rows (the Alice duplicate is gone), and Bob's hire_date is NaT --
# that's correct: "bad_date" genuinely couldn't be parsed, so it's preserved as a
# visible missing value rather than silently guessed at.
# 3행으로 줄었고(Alice 중복 제거됨), Bob의 hire_date는 NaT -- 이건 정상입니다:
# "bad_date"는 실제로 파싱 불가능했으므로, 임의로 추측하는 대신 보이는 결측치로 보존됨.

---
# ⚠️ Common Mistakes

### Mistake 1 — Using `astype()` on data you don't fully trust yet
**English:** `column.astype(int)` converts strictly — the instant it meets ONE value it can't parse (like `"abc"`), it raises a `ValueError` and halts the entire cell, even if the other 9,999 rows were perfectly fine.  
**한글:** `column.astype(int)`은 엄격하게 변환합니다 — `"abc"`처럼 파싱할 수 없는 값을 단 하나라도 만나면, 나머지 9,999개 행이 완벽했더라도 `ValueError`를 일으키며 셀 전체를 멈춥니다.

**✅ Fix / 해결법:** 
Use `pd.to_numeric(col, errors="coerce")` or `pd.to_datetime(col, errors="coerce")` for any column you haven't already verified is clean — bad values become `NaN` / `NaT` instead of crashing the pipeline. Save `astype()` for columns you already trust.  
아직 깨끗하다고 확인하지 않은 열에는 `pd.to_numeric(col, errors="coerce")`나 `pd.to_datetime(col, errors="coerce")`를 사용하세요 — 잘못된 값이 파이프라인을 멈추는 대신 `NaN` / `NaT`가 됩니다. `astype()`은 이미 믿을 수 있는 열에만 사용하세요.

### Mistake 2 — Calling bare `dropna()` on a wide table
**English:** `df.dropna()` with no arguments drops a row if **any single column** has a missing value. On a table with 15 columns, this can silently delete most of the data even when the columns you actually care about are complete.  
**한글:** 인수 없는 `df.dropna()`는 **어느 한 열**이라도 결측치가 있으면 그 행을 삭제합니다. 열이 15개인 테이블에서는, 실제로 중요한 열이 모두 채워져 있어도 데이터 대부분이 조용히 삭제될 수 있습니다.

**✅ Fix / 해결법:**  
Always pass `subset=[...]` with the specific columns that must be non-missing for a row to be usable.  
행이 사용 가능하려면 반드시 값이 있어야 하는 구체적인 열들을 `subset=[...]`으로 항상 지정하세요.

### Mistake 3 — Mixing up `axis=0` and `axis=1` in `any()` / `all()`
**English:** `(df < 60).any(axis=0)` checks **down each column** (one True/False per column — "does this subject have any failing student?"). `(df < 60).any(axis=1)` checks **across each row** (one True/False per row — "does this student fail any subject?"). Using the wrong axis silently filters the wrong dimension entirely.  
**한글:** `(df < 60).any(axis=0)`은 **각 열을 따라 아래로** 확인합니다(열마다 True/False 하나 — "이 과목에 낙제생이 있는가?"). `(df < 60).any(axis=1)`은 **각 행을 가로질러** 확인합니다(행마다 True/False 하나 — "이 학생이 낙제한 과목이 있는가?"). 잘못된 축을 쓰면 완전히 다른 차원을 조용히 필터링하게 됩니다.

**✅ Fix / 해결법:**  
Ask "do I want one answer per column, or one answer per row?" before picking the axis — per-row (for filtering rows with `df[mask]`) is `axis=1`.  
축을 고르기 전에 "열마다 답 하나를 원하는가, 행마다 답 하나를 원하는가?"를 먼저 물어보세요 — `df[mask]`로 행을 필터링하려면 `axis=1`입니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- `to_numeric()` / `to_datetime()` with `errors="coerce"` is almost always safer than `astype()` for data you didn't create yourself — bad values become `NaN` / `NaT` instead of crashing the whole cell.  
직접 만들지 않은 데이터에는 `astype()`보다 `errors="coerce"`를 붙인 `to_numeric()` / `to_datetime()`이 거의 항상 더 안전합니다 — 잘못된 값이 셀 전체를 멈추는 대신 `NaN` / `NaT`가 됩니다.
- Always pass `subset=[...]` to both `dropna()` and `drop_duplicates()` — the bare versions consider every single column, which is rarely what you actually want.  
`dropna()`와 `drop_duplicates()` 둘 다에 항상 `subset=[...]`을 지정하세요 — 인수 없는 버전은 모든 열을 고려하는데, 이는 실제로 원하는 경우가 드뭅니다.
- Chain `.str.strip().str.lower()` (or `.str.title()`) on every text column right after loading — inconsistent casing and stray whitespace is one of the most common reasons two "identical" values fail to match.  
불러온 직후 모든 텍스트 열에 `.str.strip().str.lower()`(또는 `.str.title()`)를 체이닝하세요 — 일관되지 않은 대소문자와 불필요한 공백은 "똑같은" 두 값이 일치하지 않는 가장 흔한 원인 중 하나입니다.
- `df.isna().any().any()` (two `.any()` calls) is the one-liner for "does this table have any missing value anywhere" — the first collapses each column, the second collapses that result to one final answer.  
`df.isna().any().any()`(`.any()` 두 번)는 "이 테이블에 어디든 결측치가 있는가"를 확인하는 한 줄 코드입니다 — 첫 번째가 각 열을 축약하고, 두 번째가 그 결과를 최종 답 하나로 축약합니다.

---
# 🔗 Related Concepts

```
Selection & Filtering    (Section 4 -- Boolean indexing IS the mechanism under dropna/fillna/where/mask)
    ↓
Data Cleaning              ← you are here / 지금 여기 (Section 5)
    ↓
Column Creation           (Section 6 -- np.where picks up right where where()/mask() leave off)
    ↓
GroupBy                   (Section 7 -- clean, correctly-typed columns are what make aggregations trustworthy)
    ↓
... Merge -> Pivot -> Time Series -> BA Techniques
```

*How is today's topic connected to other concepts?*

**English:** Every cleaning tool here is Section 4's filtering applied with a purpose: `dropna()` and `drop_duplicates()` are Boolean indexing under the hood, and `where()` / `mask()` take a condition exactly like `df[condition]` does but *replace* instead of *select*. Going forward, `np.where()` in Section 6 uses the same condition syntax to *build* a column instead of cleaning one. And every later GroupBy, merge, or pivot silently assumes the columns feeding it are already clean — a `groupby("dept")` on a `dept` column with `"Sales"` and `"sales "` (trailing space) as two different groups is a cleaning bug, not a GroupBy bug.

**한글:** 여기 나온 모든 정제 도구는 목적을 가지고 적용된 4번 섹션의 필터링입니다: `dropna()`와 `drop_duplicates()`는 내부적으로 Boolean indexing이고, `where()` / `mask()`는 `df[condition]`과 정확히 같은 조건을 받지만 *선택*하는 대신 *교체*합니다. 앞으로 6번 섹션의 `np.where()`는 같은 조건 문법을 사용해서 열을 정리하는 대신 *만듭니다*. 그리고 이후의 모든 GroupBy, merge, pivot은 입력되는 열이 이미 깨끗하다고 조용히 가정합니다 — `"Sales"`와 `"sales "`(끝에 공백)를 서로 다른 그룹으로 취급하는 `dept` 열에 대한 `groupby("dept")`는 GroupBy의 버그가 아니라 정제의 버그입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** You've received a raw employee roster export: messy column names, extra whitespace, inconsistent department suffixes, salary stored as text with commas, one unparseable hire date, and one duplicated row. Build the standard cleaning pipeline before handing it to HR for headcount reporting.

**한글:** 지저분한 직원 명단 내보내기 파일을 받았습니다: 지저분한 열 이름, 불필요한 공백, 일관되지 않은 부서 접미사, 쉼표가 포함된 텍스트로 저장된 연봉, 파싱 불가능한 입사일 하나, 중복 행 하나. HR에 인원 현황 보고용으로 전달하기 전에 표준 정제 파이프라인을 구축하세요.

**To Do / 할 일**
- [x] Clean up the column names / 열 이름 정리하기
- [x] Trim and standardize the casing of text columns / 텍스트 열의 공백을 제거하고 대소문자 표준화하기
- [x] Convert salary to a real number and hire_date to a real date / salary를 진짜 숫자로, hire_date를 진짜 날짜로 변환하기
- [x] Remove the duplicate row / 중복 행 제거하기
- [x] Confirm the cleaned data's remaining missing values are the expected ones / 정리 후 남은 결측치가 예상한 것과 일치하는지 확인하기

In [18]:
import pandas as pd
from io import StringIO

raw_csv = '''Employee Name,Department ,Annual Salary,Hire Date
 minsu ,SalesTeam,"48,000,000",2022-03-15
YOUNGHEE,MarketingTeam,"52,000,000",bad_date
 minsu ,SalesTeam,"48,000,000",2022-03-15
junho,EngineeringTeam,"61,000,000",2021-11-01'''

df = pd.read_csv(StringIO(raw_csv), thousands=",")

# 1) Clean column names / 열 이름 정리
df.columns = (
    df.columns.str.lower().str.strip()
    .str.replace(" ", "_")
)

# 2) Clean text columns / 텍스트 열 정리
df["employee_name"] = df["employee_name"].str.strip().str.title()
df["department"] = df["department"].str.replace("Team", "", regex=False)

# 3) Convert types / 타입 변환
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")

# 4) Remove duplicates / 중복 제거
df = df.drop_duplicates(subset=["employee_name", "department"])

print(df)
print()
print(df.dtypes)
print()
print("Any missing values left?", df.isna().any().any())
print(df.isna().sum())

  employee_name   department  annual_salary  hire_date
0         Minsu        Sales       48000000 2022-03-15
1      Younghee    Marketing       52000000        NaT
3         Junho  Engineering       61000000 2021-11-01

employee_name               str
department                  str
annual_salary             int64
hire_date        datetime64[us]
dtype: object

Any missing values left? True
employee_name    0
department       0
annual_salary    0
hire_date        1
dtype: int64


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
Data cleaning is where EDA's findings actually get fixed. Missing values get handled with `dropna()` (remove) or `fillna()` (replace, with a strategy per column) — or, for ordered data, `ffill()` / `bfill()` / `interpolate()` fill gaps from neighboring values. `drop_duplicates(subset=[...])` removes accidental re-entries, and `to_numeric()` / `to_datetime()` with `errors="coerce"` convert messy text safely, turning anything unconvertible into `NaN` / `NaT` rather than crashing — while `astype()` stays the strict, fail-loudly option for data already trusted. The `.str` accessor cleans text columns the way `.trim().toLowerCase()` would in JS, `.any()` / `.all()` check conditions across an axis, and `.clip()` / `.where()` / `.mask()` cap or conditionally replace values without a manual loop. Chained together — column names, then text, then types, then duplicates — these tools form the standard pipeline that precedes almost every real analysis.

**한글**
데이터 정제는 EDA가 발견한 것을 실제로 고치는 단계입니다. 결측치는 `dropna()`(제거)나 `fillna()`(열별 전략으로 채우기)로 처리하고 — 순서가 있는 데이터라면 `ffill()` / `bfill()` / `interpolate()`가 이웃 값으로 빈틈을 채웁니다. `drop_duplicates(subset=[...])`는 실수로 재입력된 데이터를 제거하고, `errors="coerce"`를 붙인 `to_numeric()` / `to_datetime()`은 지저분한 텍스트를 안전하게 변환해서 변환 불가능한 값을 오류 대신 `NaN` / `NaT`로 만듭니다 — 반면 `astype()`은 이미 믿을 수 있는 데이터를 위한 엄격하고 즉시 실패하는 대안으로 남습니다. `.str` accessor는 JS의 `.trim().toLowerCase()`와 비슷한 방식으로 텍스트 열을 정리하고, `.any()` / `.all()`은 축을 따라 조건을 확인하며, `.clip()` / `.where()` / `.mask()`는 반복문 없이 값을 제한하거나 조건부로 교체합니다. 열 이름 → 텍스트 → 타입 → 중복 순서로 체이닝하면, 거의 모든 실제 분석 앞에 오는 표준 파이프라인이 됩니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Data cleaning is the (usually longest) step that turns "the data I was handed" into "the data I can trust a calculation on" — missing values filled or dropped, duplicates removed, types converted safely, text normalized, and outliers capped.

> 데이터 정제는 "받은 데이터"를 "계산을 믿을 수 있는 데이터"로 바꾸는, 보통 가장 긴 단계입니다 — 결측치를 채우거나 삭제하고, 중복을 제거하고, 타입을 안전하게 변환하고, 텍스트를 정규화하고, 이상치를 제한합니다.

---
# ❓ Review Questions

**Q1.** What's the difference between `dropna()` and `dropna(subset=["col"])`?  
**Q1.** `dropna()`와 `dropna(subset=["col"])`의 차이는 무엇인가요?

dropna() removes rows containing missing values anywhere, while dropna(subset=["col"]) only checks the specified column.  
dropna()는 결측치가 있는 행을 제거하고, subset=["col"]은 특정 열만 확인해서 그 열에 결측치가 있는 행을 제거합니다.

**Q2.** Why is `pd.to_numeric(col, errors="coerce")` usually safer than `col.astype(int)` for data you didn't create yourself?  
**Q2.** 직접 만들지 않은 데이터에 대해 `pd.to_numeric(col, errors="coerce")`가 `col.astype(int)`보다 왜 보통 더 안전한가요?

pd.to_numeric(col, errors="coerce") converts invalid values to NaN instead of stopping with an error.  
pd.to_numeric(col, errors="coerce")는 변환할 수 없는 값을 NaN으로 바꾸기 때문에 에러가 발생해서 전체 작업이 중단되는 것을 막아줍니다.

**Q3.** What's the difference between `(df < 60).any(axis=0)` and `(df < 60).any(axis=1)`?  
**Q3.** `(df < 60).any(axis=0)`와 `(df < 60).any(axis=1)`의 차이는 무엇인가요?

axis=0 checks down each column, while axis=1 checks across each row.  
axis=0은 각 열을 기준으로 확인하고, axis=1은 각 행을 기준으로 확인합니다.

**Q4.** `clip()`, `where()`, and `mask()` all touch values conditionally — what does each one actually do differently?  
**Q4.** `clip()`, `where()`, `mask()`는 모두 조건부로 값을 다루는데, 각각 실제로 어떻게 다르게 동작하나요?

clip() limits values to a range, where() keeps values when the condition is True, and mask() replaces values when the condition is True.  
clip()은 값의 범위를 제한하고, where()는 조건이 True인 값을 유지하며, mask()는 조건이 True인 값을 바꿉니다.

**Q5.** In the standard cleaning pipeline (Example 8), why does column-name cleanup usually come first, and duplicate removal usually come last?   
**Q5.** 표준 정제 파이프라인(Example 8)에서, 왜 열 이름 정리가 보통 가장 먼저 오고 중복 제거가 보통 가장 마지막에 오나요?

Column names should be cleaned first so later operations can reliably reference the correct columns. Duplicates are usually removed last because cleaning and standardizing the data first can reveal rows that are truly duplicates.  
열 이름 정리는 이후의 모든 작업에서 컬럼을 정확하게 참조하기 위해 먼저 합니다. 중복 제거는 데이터를 정리하고 표준화한 뒤 최종적으로 같은 행이 무엇인지 판단하는 것이 더 정확하기 때문에 마지막에 하는 경우가 많습니다.

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*